# 🩺 YaoBi-Skill 完美复现 · 真·Tao 在环 UI（Colab + ngrok）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/Yao-Bi-Agent/blob/claude/focused-planck-3dv9we/colab/YaoBi_Skill_Colab.ipynb)

> 沈钦荣腰痹经验规则智能体 — 在 Colab 一键复现全部功能，**前端 UI 通过后端 API 真正调用语言模型（Tao）自主选择并调用 skill、自主问诊**，并经 ngrok 暴露为公网链接。

## 与"纯静态"版的区别
旧的纯静态前端把路由/规划/追问都用**浏览器端关键词规则**模拟，"Tao 选择 / Tao 在环"只是标签。本笔记本启动 **`backend.server`**（零额外依赖的 HTTP 服务），前端改为调用真实 API：
- **智能问答**：`/api/chat` → 语言模型在受限技能集内**真实选择** skill（`route_skill`，JSON 修复 + 越界回退），再由确定性规则/挖掘数据作答。
- **自主多步**：`/api/autonomous` → 语言模型**真实规划**多步并委派子智能体。
- **Tao 自动追问**：`/api/followup_probe` → 语言模型在规则约束内**真实生成**澄清式追问（经 Output Guard）。
- **智能体协作**：`/api/collaboration` → `ReasoningAgent`/`ExperienceAgent` **真实调用 Tao**。
- **对话式智能问诊**：`/api/interview` → Tao 驱动的有限状态机——模型**抽取槽位 → FSM 判阶段/红旗 → 模型自主追问 → 信息充分后生成会诊报告**；UI「智能问诊」可在「对话式（Tao）/表单式」间切换。
- **红旗急诊转诊 + 医师确认/修订/覆盖**：问诊命中红旗（如马尾综合征：大小便障碍 + 鞍区麻木）时，FSM **立即终止常规辨证**并置 `safety_level=emergency`；Tao **推理生成结构化急诊转诊临床参考**（危险信号意义 → 就诊携带信息 → 转诊前注意 → 紧迫度评级，经 Output Guard 过滤），医师端面板可**确认 / 修订 / 覆盖**该判断，覆盖后恢复问诊——符合「系统提示、医师决策」的合规闭环。
- **深度会诊回答**：语言模型是**主推理者**——把候选证型/方剂/用药/安全等规则证据作为依据，**结合自身中医知识**输出**长篇专业的辨证论治分析**（病机→证型→治法→选方方义→加减→用药模块→风险随访），而非罗列规则要点；`/api/followup_probe` 由模型**自主提问**。
- UI 右上角显示**真实运行时**徽章；只有模型真正路由时才标 `Tao 选择 ✓`，否则如实标 `关键词回退`。

## 安全边界（不变，服务端强制）
语言模型只负责**选择/编排/改写技能**；最终诊断、完整处方、患者可执行剂量被 `patient_request_guard` 与 Output Guard 拦截，违规回退确定性规则。红旗急诊转诊建议同样经 Output Guard 过滤（禁止「自行用药」类患者自用表述），并必须由执业医师在医师端**确认/修订/覆盖**后方可闭环。

## 运行顺序
依次运行单元格：② 选 Tao 运行时（默认 **Dao1-30B 全量 FP16**，推荐 A100 80GB / H100）→ ③ ngrok → **④ 启动服务并取公网链接** → ⑤ 预热 30B → ⑥ 验证 `method=llm` 及红旗急诊+医师审核合规闭环。


## ① 克隆仓库 + 基础依赖

核心依赖仅 `pyyaml`；`pyngrok` 用于公网映射。

In [ ]:
# === ① 克隆仓库 + 基础依赖 ===
import os, sys, subprocess

REPO_URL = "https://github.com/pariskang/Yao-Bi-Agent.git"
BRANCH   = "claude/focused-planck-3dv9we"   # 含真·Tao 服务端与前端；合并后可改 "main"
BASE     = "/content" if os.path.isdir("/content") else os.getcwd()
WORKDIR  = os.path.join(BASE, "Yao-Bi-Agent")

if not os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORKDIR], check=True)
else:
    print("仓库已存在，跳过克隆。")
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6.0", "pyngrok>=7.0", "openpyxl>=3.1", "pytest>=8.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("✅ 基础安装完成。")

## ② 选择 Tao 运行时（默认 Dao1-30B **全量 FP16 推理**）+ 安装模型依赖

Dao1-30B-A3B 是 30B 参数 MoE（激活约 3B）。**全量 FP16 权重约 60GB**——推荐 A100 80GB / H100 单卡，或多卡 split；单卡 24~40GB 时 `device_map='auto'` 会自动把部分层放到 CPU/RAM/disk（能跑但慢）。无 GPU/小卡用户请在代码注释中切换 mock / 小模型 / HTTP 备选。

> 之前的 4-bit 量化路径在 30B-MoE 上会触发 `Some modules are dispatched on the CPU or the disk`（bitsandbytes 不允许混合 4-bit + CPU offload），故改为官方模型卡推荐的全量 FP16 推理。

In [ ]:
# === ② 选择 Tao 运行时 + 安装模型依赖 ===
import os, subprocess, sys

# 默认：本地 transformers 加载 CMLM/Dao1-30b-a3b，**全量推理（FP16，无量化）**——
# 即官方模型卡推荐的全精度推理方式，确保权重全程驻留 GPU，避免 4-bit/8-bit 量化下
# 部分模块被 device_map='auto' 切到 CPU/disk 导致 bitsandbytes 报错。
os.environ["TAO_BACKEND"]            = "transformers"
os.environ["TAO_MODEL_ID"]           = "CMLM/Dao1-30b-a3b"
os.environ["TAO_TORCH_DTYPE"]        = "float16"
os.environ["TAO_DEVICE_MAP"]         = "auto"
os.environ["TAO_ATTN_IMPLEMENTATION"] = "eager"
os.environ["TAO_LOAD_IN_4BIT"]       = "false"   # 关闭量化
os.environ["TAO_LOAD_IN_8BIT"]       = "false"
os.environ["TAO_MAX_NEW_TOKENS"]     = "1024"   # 深度会诊式回答需要较长输出

# —— 备选运行时（按需取消注释其一）——
# 无 GPU 验证 UI/管线（模型回复为占位）：
# os.environ.update({"TAO_BACKEND": "mock"})
# 小模型本地推理（免费 T4 可跑，非 TCM 专用 Dao1）：
# os.environ.update({"TAO_BACKEND": "transformers", "TAO_MODEL_ID": "Qwen/Qwen2.5-3B-Instruct"})
# 外部 OpenAI 兼容接口：
# os.environ.update({"TAO_BACKEND": "http", "TAO_ENDPOINT_URL": "https://your-endpoint/v1/chat/completions", "TAO_API_KEY": "sk-..."})

if os.environ["TAO_BACKEND"] == "transformers":
    # CMLM/Dao1-30b-a3b 基于 Qwen3-30B-A3B（MoE，激活仅 3B），需 transformers>=4.51 才能加载该架构。
    # 全量 FP16 推理需要 sentencepiece；不再依赖 bitsandbytes（无量化路径）。
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers>=4.51", "accelerate>=0.34", "sentencepiece"], check=True)
    import transformers as _tf
    print("transformers:", _tf.__version__)
    try:
        import torch
        if torch.cuda.is_available():
            vram = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"GPU: {torch.cuda.get_device_name(0)} · {vram:.0f} GB")
            # CMLM/Dao1-30b-a3b 是 30B-A3B MoE：完整 FP16 权重 ~60 GB。
            # 单卡 < 60GB 时 device_map=auto 会自动把部分层放到 CPU/RAM/disk——
            # 加载会成功但推理会很慢；最佳是 80GB A100/H100 单卡，或多卡 split。
            if "30b" in os.environ["TAO_MODEL_ID"].lower() and vram < 60:
                print(f"⚠️ 显存 {vram:.0f}GB < 60GB——FP16 全量加载会触发 CPU/RAM offload，推理较慢。")
                print("   推荐：A100 80GB / H100；或切换上面的 mock / 小模型 / HTTP 备选。")
        else:
            print("⚠️ 未检测到 GPU；30B 全量推理需 GPU。请切 GPU 运行时，或改用 mock/http 备选。")
    except Exception as e:
        print("torch 检查跳过：", e)
print("Tao 配置:", {k: os.environ.get(k) for k in ("TAO_BACKEND", "TAO_MODEL_ID", "TAO_TORCH_DTYPE", "TAO_LOAD_IN_4BIT")})

## ③ 配置 ngrok authtoken

免费注册后到 [Your Authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) 复制（输入不回显）。

In [ ]:
# === ③ 配置 ngrok authtoken（不回显）===
from pyngrok import ngrok
from getpass import getpass

_token = getpass("粘贴你的 ngrok authtoken：").strip()
if _token:
    ngrok.set_auth_token(_token)
    print("✅ ngrok authtoken 已配置。")
else:
    print("⚠️ 未输入 token；请先配置后再运行第 ④ 步。")

## ④ ★ 启动服务器（UI + 真·Tao API）并映射公网

`backend.server` 同源提供静态 UI 与 `/api/*`。打开链接后右上角应显示绿色 **Tao 在线** 徽章；聊天/问诊/协作会真正调用语言模型。

In [ ]:
# === ④ 启动服务器（UI + 真·Tao API）并经 ngrok 映射公网 ===
import os, sys, time, subprocess
from pyngrok import ngrok

PORT = 8000
try:
    server_proc.terminate()      # 重复运行：先关旧进程
except Exception:
    pass
try:
    ngrok.kill()
except Exception:
    pass

server_proc = subprocess.Popen([sys.executable, "-m", "backend.server", "--port", str(PORT)], env=os.environ.copy())
time.sleep(2.5)
public_url = ngrok.connect(PORT).public_url
print("=" * 66)
print("✅ YaoBi-Skill 全功能 UI（真·Tao 在环）已上线")
print("🔗 打开：", public_url)
print("=" * 66)
print("· 打开后点 ngrok 的 \"Visit Site\"；右上角应显示绿色「Tao 在线」徽章。")
print("· 首次对话会触发 30B 模型下载与加载（较慢）——建议先运行第 ⑤ 步预热。")
print("· 后端运行时:", {k: os.environ.get(k) for k in ("TAO_BACKEND", "TAO_MODEL_ID", "TAO_TORCH_DTYPE")})

## ⑤ 预热模型（首次会下载并加载 `CMLM/Dao1-30b-a3b`，**全量 FP16**，较慢）

触发服务端按全量 FP16 加载模型，避免首次在 UI 里对话时长时间等待。30B 权重约 60 GB，首次下载耗时较长。

In [ ]:
# === ⑤ 预热模型 ===
import json, urllib.request

req = urllib.request.Request(f"http://127.0.0.1:{PORT}/api/warmup", data=b"{}", headers={"Content-Type": "application/json"})
try:
    res = json.load(urllib.request.urlopen(req, timeout=2400))   # 30B 首次加载可能数十分钟
    print("warmup ok:", res.get("ok"), "| ms:", res.get("ms"))
    print("runtime:", res.get("tao"))
    print("reply:", res.get("reply_preview") or res.get("reason"))
except Exception as e:
    print("预热超时/失败：", e)
    print("（也可直接在 UI 发起一次对话来触发加载；或在第 ② 步改用 mock/小模型/HTTP 备选。）")

## ⑥ 验证：语言模型真的在驱动技能选择

`method=llm` 且 `语言模型状态=accepted` 即代表模型真实路由（而非关键词回退）。本单元还验证**红旗急诊转诊（Tao 生成转诊建议）+ 医师确认/覆盖**的合规闭环。

In [ ]:
# === ⑥ 验证 ===
import json, urllib.request, urllib.error

def _post(path, body):
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}{path}", data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    try:
        return json.load(urllib.request.urlopen(req, timeout=900))
    except urllib.error.HTTPError as e:           # surface the server's JSON error body
        print("HTTP", e.code, "body:", e.read().decode("utf-8", "ignore")[:500])
        raise

print("HEALTH:", json.load(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health"))["tao"])
t = _post("/api/chat", {"question": "患者青年女性，跌扑后腰肌劳损，遇冷加重，舌淡红，苔薄白，脉细，什么证型、用什么方？", "tags": [], "doctor_mode": True})["turn"]
print("CHAT 路由 method =", t["method"], "| 路由状态 =", t["llm_routing"]["status"], "| 调用技能 =", t["skills"])
print("回答来源 =", t.get("answer_source"), "| 模型在环 =", t["used_llm"], "| 长度 =", len(t["answer"]))
print("---- 深度会诊回答（前 600 字）----")
print(t["answer"][:600])
a = _post("/api/autonomous", {"question": "是什么证型、用什么方、有什么风险？", "tags": ["lower_limb_numbness", "cold_aggravation", "dark_tongue"], "doctor_mode": True})["turn"]
print("\nAUTO 规划方式 =", a["plan_method"], "| 计划 =", [p["label"] for p in a["plan"]])

# 对话式智能问诊（Tao 驱动 FSM）：抽取→判阶段→自主追问
_post("/api/interview", {"session_id": "nb", "reset": True})
iv = _post("/api/interview", {"session_id": "nb", "message": "腰痛半年，弯腰加重，左腿发麻，遇冷加重，舌暗，苔白腻"})
print("\nINTERVIEW 阶段 =", iv["state"], "| 安全 =", iv["safety_level"], "| 候选证候 =", [p["pattern"] for p in iv["candidate_patterns"][:3]])
print("Tao 自主追问 =", iv["message"][:120])

# —— 红旗急诊转诊 + 医师确认/修订/覆盖（合规闭环）——
# 1) 命中马尾综合征红旗 → 立即终止常规辨证 + Tao 生成急诊转诊建议 + 待医师审核
_post("/api/interview", {"session_id": "rf", "reset": True})
rf = _post("/api/interview", {"session_id": "rf", "message": "腰痛伴大小便失禁，会阴部麻木"})
print("\n[红旗] 阶段 =", rf["state"], "| 安全级别 =", rf["safety_level"], "| 终止问诊 done =", rf["done"], "| 需医师审核 =", rf["physician_review_required"])
print("[红旗] 危险信号 =", rf["red_flags"])
print("[红旗] Tao 急诊转诊参考（前 220 字）：\n", (rf.get("referral_tao_guidance") or "（模型输出未通过 Output Guard，已回退确定性规则）")[:220])
# 2) 医师确认 → 背书急诊转诊建议（done 保持 True）
cf = _post("/api/interview", {"session_id": "rf", "review_action": "confirm", "physician_notes": "已联系120转运"})
print("\n[医师确认] 审核状态 =", cf["physician_review"]["status"], "| 医师备注 =", cf["physician_review"]["physician_notes"])
# 3) 另一会话演示医师覆盖 → 清除红旗并恢复问诊（系统提示、医师决策）
_post("/api/interview", {"session_id": "rf2", "reset": True})
_post("/api/interview", {"session_id": "rf2", "message": "腰痛伴大小便失禁，会阴部麻木"})
ov = _post("/api/interview", {"session_id": "rf2", "review_action": "override", "override_reason": "患者描述有误，实际无失禁/无鞍区麻木"})
print("\n[医师覆盖] 审核状态 =", ov["physician_review"]["status"], "| 恢复后阶段 =", ov["state"], "| 安全级别 =", ov["safety_level"], "| 继续问诊 done =", ov["done"])

## ⑦ 后端规则管线 CLI（纯确定性，不加载模型）

这些是确定性规则引擎，不触发 Tao 加载；真·Tao 调用见上面的 UI 与第 ⑥ 步。

In [ ]:
# === ⑦ 规则管线 CLI（纯确定性）===
CASE = "患者女，68岁，腰痛反复5年，加重1月，伴下肢麻木，畏寒，舌暗苔白腻，脉细缓，既往骨质疏松。"
print(">>> 规则管线报告\n")
!python -m backend.main --text "{CASE}"
print("\n>>> 自主多步（确定性规划）\n")
!python -m backend.main --ask "这个病人是什么证型、用什么方、有什么风险？" --autonomous

## ⑧ 运行全量测试（含真·Tao 服务端用例）

覆盖：服务端 `/api/*` 真实语言模型路由/规划/追问/协作、患者拦截、量化配置、前端 API 接线、红旗硬门控、Output Guard 等。

In [ ]:
# === ⑧ 全量测试 ===
!python -m pytest -q

## ⑨ xlsx 脱敏医案规则挖掘（纯函数演示）

仓库已内置脱敏挖掘产物 `frontend/mined_rules.js`，UI 的总览/挖掘/证据页直接读取。下面用内存中的脱敏 case 字典演示纯函数管道（不落盘）。

In [ ]:
# === ⑨ 脱敏挖掘（纯函数）===
from backend.mining.xlsx_case_miner import mine_cases, build_rule_candidates
cases = [
    {"row_id": 2, "sex": "女", "age": 68, "age_band": "60s",
     "symptom_tags": ["lower_limb_numbness", "cold_aggravation", "elderly"],
     "zheng": "气血痹阻证", "tcm_diseases": ["腰痹"], "western_dx": ["腰椎间盘突出"],
     "herbs": ["独活", "桑寄生", "当归", "细辛", "牛膝"], "herb_doses": {"细辛": 3.0}, "has_prescription": True, "osteoporosis_history": False},
    {"row_id": 3, "sex": "男", "age": 72, "age_band": "70s",
     "symptom_tags": ["lower_limb_numbness", "elderly", "osteoporosis"],
     "zheng": "肝肾不足证", "tcm_diseases": ["腰痹"], "western_dx": ["骨质疏松"],
     "herbs": ["独活", "桑寄生", "杜仲", "牛膝"], "herb_doses": {"杜仲": 12.0}, "has_prescription": True, "osteoporosis_history": True},
]
mined = mine_cases(cases)
print("证型分布:", mined["dataset_stats"]["zheng_distribution"])
print("签名方剂命中:", [h["formula"] for h in mined["formula_signature_hits"]])
print("候选规则:", [r["rule_id"] for r in build_rule_candidates(mined)[:5]])

## ⑩ 清理（停止服务与隧道）

In [ ]:
# === ⑩ 清理 ===
try:
    from pyngrok import ngrok; ngrok.kill(); print("已关闭 ngrok 隧道。")
except Exception as e:
    print("ngrok:", e)
try:
    server_proc.terminate(); print("已停止后端服务。")
except Exception as e:
    print("server:", e)